# Лабораторная работа 2. Введение в машинное обучение: knn, ann, оптимизация

Результат лабораторной работы − отчет. Мы предпочитаем принимать отчеты в формате ноутбуков IPython (ipynb-файл). Постарайтесь сделать ваш отчет интересным рассказом, последовательно отвечающим на вопросы из заданий. Помимо ответов на вопросы, в отчете также должен быть код, однако чем меньше кода, тем лучше всем: нам − меньше проверять, вам — проще найти ошибку или дополнить эксперимент. При проверке оценивается четкость ответов на вопросы, аккуратность отчета и кода. Текущая лабораторная работа будет проверяться в **половину баллов!**, если один и тот же код будет копироваться между заданиями. Пишите модульный код, который можно будет переиспользовать между заданиями.

За текущую лабораторную работу можно получить от 33 до 36 баллов. 3 балла будут дополнительными, они не пойдут в общую сумму.

Мы уверены, что выполнение лабораторных работ занимает значительное время, поэтому не рекомендуем оставлять их на последний вечер перед сдачей.

Обратите внимание, что мы не ставим оценку за просто написанный код, корректная работоспособность которого не подтверждена экспериментами.


## 0. Baseline

Рассмотрим задачу оценки риска заболевания сахарным диабетом.

На эту задачу мы посмотрим со стороны страховой компании. Если человек перестанет проходить обследования и страховая так и не узнает, развился ли у него диабет, то и расходов, связанных с его заболеванием, не будет, т.е. можно считать, что такой человек остался здоров. То же касается людей, у которых заболевание впервые обнаружат более, чем через 5 лет.

Чтобы рассчитать математическое ожидание затрат на лечение клиента, страховая хочет получить в качестве результата работы модели непосредственно вероятность того, что у человека, не страдающего от заболевания, оно разовьётся в течение 5 лет. Поэтому в качестве метрики качества была выбрана бинарная кросс-энтропия (она же logloss) между предсказанными вероятностями и истинными метками классов:

$$\text{crossentropy}(y, p) = -\frac{1}{N}\sum_{i=1}^N\left[y_i\log(p_i) + (1-y_i)\log(1-p_i)\right]$$

**Задание 1 (0.5 балла). Загрузка данных, тривальный препроцессинг.**

- Загрузите [данные задачи о сахарном диабете](https://disk.yandex.ru/d/LIkoBl_WkWarTw), прочитайте описание признаков.
- Обратите внимание, что часть информации о клиентах неизвестна на момент заключения договора. Соответствующие признаки отсутствуют в X_test.csv.
- Заполните пропуски в данных. Для этого могут пригодиться методы из sklearn.impute или pandas.DataFrame.fillna.
- По желанию, используйте любой препроцессинг данных, добавляйте новые признаки и т.п. Ваша задача — добиться сходимости и высокого качества полученных моделей.
- Разбейте обучающую выборку на train и test, которые будете использовать для оценки всех построенных моделей в лабораторной работе. При желании использовать для оценки качества кросс-валидацию необходимо проконтролировать, чтобы для всех моделей использовались одни и те же разбиения.

In [ ]:
# TODO: code here

**Задание 2 (1-4 балл). Нетривиальный препроцессинг и исскусство работы с данными**

Основное количество баллов здесь равно 1 и ещё 3 дополнительных, которые не идут в общую сумму.

Данное задание творческое: исследуйте датасет, ищите зависимости между данными, обогащайте датасет новыми фичами, не используя модели, стройте графики распределений, заполняйте данные по ним и т.д. и т.п. Чем нетривиальнее зависимость, найденная в этом пункте, тем выше балл!

In [ ]:
# TODO: code here

**Задание 3 (0.5 балла).  Бэйзлайн** – константное предсказание.

Как понять, работает ли та или иная модель, если сравнить метрику не с чем? Чтобы было с чем сравнивать, соберём простой бэйзлайн: предскажем всем клиентам одну и ту же вероятность заболеть в течение 5 лет. Какое значение надо предсказать, чтобы минимизировать кросс-энтропию? Оцените качество такого предсказания.

In [ ]:
# TODO: code here

## 2. KNN

В данной части вам предстоит реализовать метод knn.

**Задание 4 (3 балла).  plain knn**


Реализовать наивный knn, который запоминает весь датасет и предсказывает метки классов по *n_neighbors* ближайшим соседям по метрике *metric* и весам *weights*. Предсказанием здесь будет наиболее частая метка классов среди соседей.

Параметры n_neighbors, metric, weights должны работать по аналогии с `KNeighborsClassifier` в sklearn.

Возможные параметры:
- `metric: {euclidean, manhattan, cosine, l1}`
- `weights: {uniform, distance}` 

Важно! В решении нельзя использовать готовые реализации метрик, например из sklearn, если не сказано обратного. Нужно их запрогать руками. В конце задачи сравните полученные результаты с реализацией sklearn. (не забудьте указать в реализации sklearn `algorithm=brute`, чтобы симулировать одинаковые вычисления)! Так же постарайтесь вычислять попарные расстояния как можно реже, например, для вычисления веса при `uniform=weights` дважды считать веса не нужно.

Помимо этого, ваша реализация должна поддерживать одновременное предсказание для батча сэмплов. Другими словами, метод `predict()` должен принимать на вход массив размера `[batch_size, dim]`. В процессе реализации постарайтесь по максимуму использовать векторизированные операции и **не делать питонячьи циклы по сэмплам из батча**.

In [ ]:
# TODO: code here

**Задание 5 (1 балл). knn с батчами**


Бывает ситуация, когда все вектора в датасете разом не помещаются в память. В таком случае подсчет расстояний до объектов в выборке можно проводить *батчами*. Тогда при подсчете расстояний очередной батч сэмплов из обучающей выборки подгружается в память "на лету". Сгружать датасет на диск мы не будем, а вот попробовать симулировать подобное ограничение сможем!

Добавьте в реализацию выше параметр `max_train_batch_size` -- максимальный размер сэмплов в обучающей выборке, которые одномоментно могут участвовать в подсчете расстояний. Другими словами, если `max_train_batch_size=10`, то расстояния до объектов обучающей выборки нужно считать "десятками", т.е. расстояния от объекта до первых десяти сэмплов, потом до вторых и тд. Критерий правильности решения -- при вызове функции расчета попарных расстояний, размер подмассива обучающей выборки не должен превышать `max_train_batch_size`.

В конце сравните результаты, они не должны изменяться в зависимости от `max_train_batch_size`.

In [ ]:
# TODO: code here

**Задание 6 (3 балла). реализация ann**

Считать для очередного сэмпла расстояния до всех объектов обучающей выборки, чтобы потом сделать предсказания, бывает дорого. По дефолту KNN из sklearn использует приблеженные методы: BallTree или KDTree. Реализуйте на выбор алгоритмы [BallTree](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.BallTree.html#sklearn.neighbors.BallTree) или [KDTree](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KDTree.html#sklearn.neighbors.KDTree).

Продемонстрируйте, что ваш результат поиска ближайших соседей вашим алгоритмом эксвивалентен с некоторой погрешностью c вашей реализации plain KNN.


In [ ]:
# TODO: code here

**Задание 7. (4 балла) ускорение вычисления близостей вложений.**

Как известно, во время стандартного сценария инференса мы упираемся в поиск top-k похожих. Но бывают задачи, когда нам надо для выделенного множества товаров/документов/айтемов посчитать близости с заданными сущностями. Самый очевидный пример такого сценария - это подсчет фичей в [информационном поиске](https://en.wikipedia.org/wiki/Information_retrieval) между вектором запросом и векторами, которые представляют айтемы в выдаче.

В данном задании вам предлагается углубиться именно в сценарий ускоренного расчета косинусных близостей на основе квантинизаци. Предложите и опишите несколько вариантов такой квантинизации (из открытых источников) и реализуйте квантинизацию предложенную [авторами NGT](https://medium.com/@masajiro.iwasaki/fusion-of-graph-based-indexing-and-product-quantization-for-ann-search-7d1f0336d0d0#8a43).

Продемонстрируйте, что ваш результат поиска ближайших соседей вашим алгоритмом эксвивалентен с некоторой погрешностью c вашей реализации plain KNN.

P.S. Реализацию алгоримта KMeans берите из sklearn, так как формально мы тему кластеризации еще не проходили

In [ ]:
# TODO: code here

**Задание 8. (2 балла) замер размена качества на скорость работы**

Продемонстрируйте, какой у вас получается размен recall от результата работы plain knn на скорость работы алгоритмов (из заданий 6 и 7) на этапе получения top-K сосоедей.

Варьируя параметры, которые отвечают за точность приближения - ваши алгоритмы должны работать быстрее, но их recall должен падать. Будет плюсом, если вы также замерите не только время работы алгоритмов, но и время построения всех структур для поиска.




In [ ]:
# TODO: code here

# 2. Логистическая регрессия и методы оптимизации

В данной части вам предстоит реализовать свои алгоритмы. Но так как у них будут разные лоссы, сравнивать их по ним будет некорректно. Поэтому в качестве метрики качества возьмем ROC-AUC.

**Задание 8 (1 балл). ROC-AUC**


Необходимо реализовать функцию расчета ROC-AUC.

Важно! Полностью правильное решение не должно иметь квадратичную сложность по количеству семплов.

In [ ]:
import numpy as np

In [ ]:
def my_roc_auc_score(y_true: np.ndarray, y_score: np.ndarray) -> float:
    # TODO: code here
    return .0

In [ ]:
from sklearn.metrics import roc_auc_score

test_1_pred = np.array([
    0.46796706, 0.52076168, 0.57961271, 0.85361132, 0.80184669,
    0.26700869, 0.77085205, 0.66930147, 0.73739676, 0.08905119,
    0.83543104, 0.20286153, 0.90183663, 0.40937635, 0.51630144,
    0.50068668, 0.59059626, 0.23382453, 0.40684045, 0.81429939     
])
test_1_true = np.array([1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1])

test_2_pred = np.array([
    28.54342069,  5.26742071, 10.43160241,  8.9613801 ,  3.32349059,
    17.39657995, 21.8405144 , 15.5825081 ,  9.98476545, 10.48585524,
    20.44766387, 27.66281023, 16.19600877, 17.09394291, 13.4002362 ,
    21.31178614,  9.69002596
])
test_2_true = np.array([0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1])

for test_true, test_score in [[test_1_true, test_1_pred], [test_2_true, test_2_pred]]:
    assert np.allclose(
        roc_auc_score(test_true, test_score),
        my_roc_auc_score(test_true, test_score),
        atol=1e-3
    )

print("All tests passed!")

## 2.1 Logistic regression and SGD

**Задание 9 (2 балла). Логистическая регрессия и стохастический градиентный спуск.**

Напишите свою модель логистической регрессии, воспользовавшись кодом с семинара. Обучите модель логистической регрессии c помощью стохастического градиентного спуска. Требование к вашей реализации следующее:
- реализовать отдельный класс SGD, который осуществляет спуск по заданному алгоритму и отдельный класс логистической регрессии
- нужно уметь сериализовывать стейт вашего градиентного спуска и продолжать спуск с него. Это нужно, в том числе, при загрузке так называемых checkpoint-ов при аварийном завершении программы, например, при нехватке оперативной памяти

Пример интерфейса вашего SGD класса:
```(python)
class MyCoolSGD:
    def __init__(self, algorithm):
        self.algorithm = algorithm
        self.state = {
            "weights": algorithm.weights,
            "gradients": np.zeros_like(algorithm.weights),
            "sgd_params": <some_params>
        }

    def save_state(self, path):
        with open(path, "wb") as f:
            pickle.dump(self.state, path)

    def load_state(self, path):
        with open(path, "wb") as f:
            self.state = pickle.load(path)
```

Продемонстрируйте, как ваша реализация SGD умеет восстанавливать стейт обучения.

В качестве критерия останова мы предлагаем использовать следующие условия:
 - евклидова норма разности текущего и нового векторов весов стала меньше, чем 1e-4
 - ограничение на число эпох обучения (например 3)

Какой вывод вы можете сделать? Удалось ли получить стабильное улучшение по сравнению с предыдущими моделями? Чем это можно объяснить?


In [ ]:
# TODO: code here

**Задание 10 (3 балла). Градиентный спуск для логистической регрессии.**

Попробуйте 2-3 эвристики из каждого пункта для градиентного спуска (например, [отсюда]((https://neerc.ifmo.ru/wiki/index.php?title=Стохастический_градиентный_спуск#.D0.AD.D0.B2.D1.80.D0.B8.D1.81.D1.82.D0.B8.D0.BA.D0.B8)), [Xavier initialization](https://paperswithcode.com/method/xavier-initialization)) и сравните, как они влияют на скорость обучения и качество:
- различный выбор начальной инициализации весов
- различный способ выбора элементов для батча

Помимо перебора эвристик отдельно переберите:
- learning_rate - коэфициент перед градиентами или размер шага
- количество эпох градиентного спуска
- размер минибатча.

Исследуйте качество оптимизируемого функционала в зависимости от номера итерации. В каждом пункте требуется построить необходимые графики скорости/качества и дать исчерпывающие выводы.


In [ ]:
# TODO: code here

**Задание 11 (2 балла). Другие лоссы классификации.**

Поддержите в своей реализации классификатора как минимум 2 [различных лосса](https://scikit-learn.org/stable/modules/sgd.html#mathematical-formulation): hinge, perceptron, modified huber, ...

В данном задании помимо изменения лосса у вас меняются и градиенты, учитывайте это при использовании SGD.

Рассмотрите, как меняется качество предсказания вашего классификатора. Какие выводы вы можете сделать?

In [ ]:
# TODO: code here

Как известно, $L_1$-регуляризация линейной регрессии (LASSO) зануляет коэффициенты модели при стремлении параметра регуляризации в бесконечность.
В случае линейной регрессии, хотя доказать это непросто, можно легко визуализировать. Функция потерь в логистической регресиии более сложная (не забываем, что мы оптимизируем не сам лосс, а сумму значений loss-function по всем $x_i$).

Теорию по регуляризации можете вспомнить из соответсвующей части учебника: https://education.yandex.ru/handbook/ml/article/linear-models#regulyarizacziya

**Задание 12 (0.5 балла). Regularization. Support l1.**

Поддержите l1 регуляризацию для вашей реализации логистической регрессии


In [ ]:
# TODO: code here


**Задание 13 (0.5 балла). Regularization. Support l2.**

Поддержите l2 регуляризацию для вашей реализации логистической регрессии


In [ ]:
# TODO: code here


**Задание 14 (1.5 балл). Regularization. Comparison.**

Перебрите по сетке значение коэффициента при $L_2$-регуляризаторе и $L_1$ регуляризаторе, замеряя при этом качество модели, время обучения (с каким регуляризатором оптимизация быстре сходится - проверяем на фиксированном методе оптимизации, можно построить графики для нескольких методов и сравнить), а также визуализируя барами коэффициенты вектора $w$.  Зануляются ли коэффициенты при $L_2$ регуляризаторе? А при $L_1$-регуляризаторе. Если да, то какие коэффициенты зануляются первыми? Какие признаки модель считает наиболее значимыми?

In [ ]:
# TODO: code here

*!!Методы моментов должны поддерживать сериализацию/десериализацию стейта обучения. Без этого за задание будут 0 баллов!!*

Теорию можете найти [здесь](https://education.yandex.ru/handbook/ml/article/optimizaciya-v-ml#ispolzovanie-informaczii-o-predydushhih-shagah).

**Задание 15 (1 балл). Simple Momentum.**

Реализуйте метод оптимизации *simple momentum* и продемонстрируйте, как метод моментов поддерживает сериализацию/десериализацию стейта обучения.

$$
v_{k+1} = \beta_k v_k - \alpha_k \nabla f(x_k) x_{k+1} \\
x_{k+1} = x_k + v_{k+1}
$$

$\beta_k = 0.9$

**Задание 16 (1 балл). Nesterow Momentum.**

Реализуйте метод оптимизации *nesterow momentum* и продемонстрируйте, как метод моментов поддерживает сериализацию/десериализацию стейта обучения.

$$
v_{k+1} = \beta_k v_k - \alpha_k \nabla f(x_k) x_{k+1} \\
x_{k+1} = x_k + v_{k+1}
$$

$\beta_k = 0.9$

**Задание 17 (0.5 балла). Сравнение методов момента**

Сравните реализованные вами методы моментов с ванильно реализацией SGD с точки зрения качества/скорости сходимости. Переберите параметр $\beta$, предоставьте исчерпывающие выводы. Какой алгоритм лучше показал себя Simple Momentum, Nesterow Momentum, ванильный SGD? Почему так получилось?

In [ ]:
# TODO: code here

Адаптивные методы подбора шага должны поддерживать сериализацию/десериализацию стейта обучения. Теорию можете найти [здесь](https://education.yandex.ru/handbook/ml/article/optimizaciya-v-ml#adaptivnyj-podbor-razmera-shaga).

**Задание 18 (1 балл). Adagrad.**

Реализуйте метод оптимизации *Adagrad* и продемонстрируйте, как метод моментов поддерживает сериализацию/десериализацию стейта обучения.

$\gamma=0.99, \varepsilon = 1e-8$

**Задание 19 (1 балл). RMSProp.**

Реализуйте метод оптимизации *RMSProp* и продемонстрируйте, как метод моментов поддерживает сериализацию/десериализацию стейта обучения.

$\gamma=0.99, \varepsilon = 1e-8$

**Задание 20 (0.5 балла). Сравнение адаптивных методов подбора шага**

Сравните реализованные вами методы адаптивного подбора шага с ванильно реализацией SGD с точки зрения качества/скорости сходимости. Переберите параметр $\gamma$, предоставьте исчерпывающие выводы. Какой алгоритм лучше показал себя Adagrad, RMSProp, ванильный SGD? Почему так получилось?

In [ ]:
# TODO: code here

**Задание 21 (1.5 балла). Реализуйте метод Adam.**

Метод Adam является смесью метода моментов и адаптивного подбора шага. Реализуйте данный метод SGD и сравните его результаты с лучшим методом SGD до этого. Какие они, почему так получилось?

В качестве рефернсного псевдокода можно обратиться к [доке пайторча](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html) (в нашей реализации `amsgrad=False` всегда) или исходной статье.

Важно! В большенстве учебников реализацию адама делают без bias correction term, в лабе нужно этот шаг учесть (в псевдокоде пайторча и статье он есть).

In [23]:
# TODO: code here

**Задание 22 (1 балл). Вывод**

Опишите в свободной форме вывод по вашей работе. Процитируйте дополнительные материалы, если их использовали, которые помогли лучше разобраться в лабораторной. Если решали с GPT моделями - то помогли ли они вам в каких-либо заданиях и как. Порефлексируйте над результатами: что получилось, что нет и почему.

In [ ]:
# TODO: code here